# Exploration du dataset de noms

Objectif : vérifier librement la pertinence des noms de `data/interim/names.jsonl`.

Trois modes d'exploration : tirage aléatoire, audit d'une source, recherche par motif. Les dernières sections affichent des familles d'anomalies potentielles à juger à l'œil — **aucune exclusion n'est faite ici**.

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

path = Path("../data/interim/names.jsonl")  # cwd du kernel = notebooks/
names = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
len(names), names[:3]

## Carte des sources

In [ ]:
volumes = Counter(n["source"] for n in names)
print(f"{len(names)} noms, {len(volumes)} sources")
print()
for source, count in volumes.most_common(25):
    print(f"{count:>5}  {source}")

## Exploration libre

Chaque cellule tire un nouvel échantillon à chaque exécution : relancer la cellule suffit pour une nouvelle fenêtre sur le dataset.

In [ ]:
N = 30  # taille du tirage ; relancer la cellule pour un nouvel échantillon

for n in random.sample(names, min(N, len(names))):
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
SOURCE = "NotreDameDeParis"  # fragment du nom de source, insensible à la casse
MAX_SHOWN = 60  # au-delà, tirage aléatoire dans la source

pool = [n for n in names if SOURCE.lower() in n["source"].lower()]
print(f"{len(pool)} noms dans les sources matchant {SOURCE!r}")
print()
shown = pool if len(pool) <= MAX_SHOWN else random.sample(pool, MAX_SHOWN)
for n in sorted(shown, key=lambda n: n["canonical"].lower()):
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
PATTERN = r"^Saint"  # regex appliquée au nom canonique
LIMIT = 80

rx = re.compile(PATTERN)
hits = [n for n in names if rx.search(n["canonical"])]
print(f"{len(hits)} noms matchant /{PATTERN}/")
print()
for n in hits[:LIMIT]:
    print(f"{n['canonical']:<45}  {n['source']}")

## Anomalies potentielles

Vues de contrôle, à juger à l'œil. Une source qui sort souvent ici s'audite avec la cellule `SOURCE` ci-dessus. Les exclusions éventuelles seront tranchées en Phase 2 (normalize) — ce notebook ne modifie jamais `names.jsonl`.

In [ ]:
# Génériques / collectifs : probablement pas des noms propres de personnages
GENERIC_RX = re.compile(
    r"^(?:un |une |deux |trois |plusieurs |famille |peuple |groupe )",
    re.IGNORECASE,
)
generic = [n for n in names if GENERIC_RX.search(n["canonical"])]
print(f"{len(generic)} noms génériques / collectifs")
print()
for n in generic:
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
# Rôles d'opéra listés en anglais dans les pages Wikipédia francophones
ENGLISH_TOKENS = {
    "the",
    "of",
    "a",
    "an",
    "old",
    "man",
    "girl",
    "boy",
    "shepherd",
    "voice",
    "first",
    "second",
    "third",
    "from",
    "king",
    "queen",
    "servant",
    "soldier",
    "child",
    "milk",
    "dream",
    "bride",
    "mother",
}


def looks_english(name: str) -> bool:
    tokens = re.split(r"[\s'\u2019-]+", name.lower())
    return any(token in ENGLISH_TOKENS for token in tokens)


english = [n for n in names if looks_english(n["canonical"])]
print(f"{len(english)} noms anglophones probables ; principales sources :")
print(Counter(n["source"] for n in english).most_common(10))
print()
for n in english:
    print(f"{n['canonical']:<45}  {n['source']}")

In [ ]:
# Noms anormalement longs : souvent des listes de rôles plutôt qu'un nom
LENGTH_LIMIT = 40

lengths = sorted(len(n["canonical"]) for n in names)
print(
    f"longueur : min {lengths[0]}, médiane"
    f"  {lengths[len(lengths) // 2]}, max {lengths[-1]}"
)
print()
long = [n for n in names if len(n["canonical"]) > LENGTH_LIMIT]
print(f"{len(long)} noms de plus de {LENGTH_LIMIT} caractères :")
print()
for n in sorted(long, key=lambda n: -len(n["canonical"])):
    print(f"{len(n['canonical']):>3}  {n['canonical']}  <- {n['source']}")

In [ ]:
# Titres sociaux intégrés au nom : informatif pour Phase 2 (normalize)
TITLES = (
    "roi",
    "reine",
    "marquis",
    "marquise",
    "duc",
    "duchesse",
    "comte",
    "comtesse",
    "vicomte",
    "vicomtesse",
    "baron",
    "baronne",
    "abbé",
    "madame",
    "mademoiselle",
    "monsieur",
    "maître",
    "saint",
    "sainte",
)
title_rx = re.compile(r"\b(?:" + "|" + ".join(TITLES) + r)\b", re.IGNORECASE)
titled = [n for n in names if title_rx.search(n["canonical"])]
print(f"{len(titled)} noms contenant un titre social ; distribution :")
print(
    Counter(
        m.lower() for n in titled for m in title_rx.findall(n["canonical"])
    ).most_common()
)
print()
for n in random.sample(titled, min(25, len(titled))):
    print(f"{n['canonical']:<45}  {n['source']}")